<a href="https://colab.research.google.com/github/Khasanov01/jaydariGPT/blob/main/jaydari_gpt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install --only-binary=:all: transformers torch bitsandbytes datasets peft trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 48.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 60.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [trl]


In [ ]:
from transformers import AutoTokenizer, BitsAndBytesConfig, AutoModelForCausalLM, TrainingArguments
import torch
from datasets import load_dataset
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer

In [ ]:
model_id='TinyLlama/TinyLlama-1.1B-Chat-v1.0'
tokenizer=AutoTokenizer.from_pretrained(model_id)

#print('vocab size:', tokenizer.vocab_size)
#print('special_tokens_map:', tokenizer.special_tokens_map)

#quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

bnb_config
model=AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map='auto',
    dtype=torch.float16
)

In [ ]:
# Before Fine-tuning
prompt = "Explain what a tokenizer is."

inputs = tokenizer(prompt, return_tensors="pt").to(model.device) # GPU, CPU

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=True,
        temperature=0.7
    )

print(tokenizer.decode(output_ids[0], skip_special_tokens=True))

#print(model)
first_block=model.model.layers[0]
print("first_block:", first_block)
print("========")
print(first_block.self_attn)
print("========")
print(model.config)

In [ ]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

total_params = count_parameters(model)
print(f"Total parameters (including frozen 4-bit): {total_params:,}")

Total parameters (including frozen 4-bit): 615,606,272


# datasets library | load_dataset
## instruction tuning

In [ ]:

dataset = load_dataset("yahma/alpaca-cleaned", split="train")
dataset

In [ ]:
def generate_prompt(example):
    instruction = example['instruction']
    input_text = example['input']
    output_text = example['output']

    if input_text:
        return (
            f"### Instruction:\n"
            f"{instruction}\n\n"
            f"### Input:\n"
            f"{input_text}\n\n"
            f"### Response:\n"
            f"{output_text}"
        )
    else:
        return (
            f"### Instruction:\n"
            f"{instruction}\n\n"
            f"### Response:\n"
            f"{output_text}"
        )

#generate_prompt(dataset[0])

def formatting_func(example):
  return{'text': generate_prompt(example)}

dataset=dataset.map(formatting_func)

Map:   0%|          | 0/51760 [00:00<?, ? examples/s]

In [ ]:
dataset[0]['text']

In [ ]:
dataset=dataset.select(range(7000))

In [ ]:
dataset=dataset.shuffle(seed=42)

In [ ]:
# [7, 3, 2, 8, 5, 6, 9, 4, 0, 1]
# [7, 3, 2, 8, 5, 6, 9, 4, 0, 1] EXACT SAME ORDER

In [ ]:
# [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
# [10, 231, 342, 3453, 3464, 5123, 6456, 7, 8, 9]

In [ ]:
dataset

Dataset({
    features: ['output', 'input', 'instruction', 'text'],
    num_rows: 7000
})

In [ ]:
# Full Fine-tuning =>
# Cheap Fine-tuning =>
# PEFT => parameter efficient fine-tuning
# OOM => Out Of Memory

In [ ]:

lora_config = LoraConfig(
    r=8, # rank
    lora_alpha=16,
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj"],
)

In [ ]:
model = get_peft_model(model, lora_config)

In [ ]:
model.print_trainable_parameters()

trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023


In [ ]:
#LoRA
#QLoRA

In [ ]:
training_args = TrainingArguments(
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4, # VRAM
    num_train_epochs=2, # overfit
    logging_steps=20,
    output_dir="./jaydari_gpt",
    save_strategy="epoch",
    fp16=True,
    report_to="none"
)

In [ ]:
# 1. Convert all trainable LoRA adapter weights to float32
for p in model.parameters():
    if p.requires_grad:
        p.data = p.data.to(torch.float32)

# 2. Turn off the GradScaler that is looking for bfloat16
training_args.fp16 = False

In [ ]:
# SFTTrainer
trainer=SFTTrainer(
    model=model,
    train_dataset=dataset,
    formatting_func=lambda x: x['text'],
    args=training_args,
)
trainer.train()
model.save_pretrained('jaydari_gpt')
tokenizer.save_pretrained('jaydari_gpt')

Applying formatting function to train dataset:   0%|          | 0/7000 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/7000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/7000 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/7000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/7000 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/7000 [00:00<?, ? examples/s]

Step,Training Loss
20,1.522407
40,1.575810
60,1.316646
80,1.368152
100,1.303759
120,1.323515
140,1.349285
160,1.244384
180,1.309380
200,1.217852


('jaydari_gpt/tokenizer_config.json',
 'jaydari_gpt/chat_template.jinja',
 'jaydari_gpt/tokenizer.json')

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

base_model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer_base = AutoTokenizer.from_pretrained(base_model_id)
model_base = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    device_map="auto"
)

model_base.eval()

In [ ]:
prompt = "Explain what machine learning is in simple words."

inputs_base = tokenizer_base(prompt, return_tensors="pt").to(model_base.device)

with torch.no_grad():
    output_base = model_base.generate(
        **inputs_base,
        max_new_tokens=120,
        temperature=0.7,
        do_sample=True
    )

print("====== BASE MODEL OUTPUT ======")
print(tokenizer_base.decode(output_base[0], skip_special_tokens=True))

[transformers] Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


====== BASE MODEL OUTPUT ======
Explain what machine learning is in simple words.


# FINE-TUNED MODEL INFERENCE


In [ ]:
!pip install -U torchao
import peft.import_utils
peft.import_utils.is_torchao_available = lambda: False

model_path = "jaydari_gpt"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    device_map="auto"
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 21.8 MB/s  0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

W1116 11:18:08.846000 364 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Loading weights:   0%|          | 0/88 [00:00<?, ?it/s]

In [ ]:
prompt = """### Instruction:
Explain what machine learning is in simple words.

### Response:
"""

inputs_ft = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    output_ft = model.generate(
        **inputs_ft,
        max_new_tokens=120,
        temperature=0.7,
        do_sample=True
    )

print("====== FINE-TUNED MODEL OUTPUT ======")
print(tokenizer.decode(output_ft[0], skip_special_tokens=True))

[transformers] Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


====== FINE-TUNED MODEL OUTPUT ======
### Instruction:
Explain what machine learning is in simple words.

### Response:
Machine learning, also known as artificial intelligence, is a field that studies how a computer can learn and improve its algorithms through experience. It is an exciting branch of technology and helps businesses make data-driven decisions by analyzing and understanding large amounts of data.

In simple terms, machine learning is the process of making a computer learn from data. It involves algorithms that can predict outcomes based on the data collected. This process involves learning from data and using that information to improve the computer's decision-making capabilities.

Machine learning can be used in a variety of


In [ ]:
!zip -r jaydari_gpt.zip jaydari_gpt

  adding: jaydari_gpt/ (stored 0%)
  adding: jaydari_gpt/README.md (deflated 44%)
  adding: jaydari_gpt/tokenizer.json (deflated 85%)
  adding: jaydari_gpt/tokenizer_config.json (deflated 46%)
  adding: jaydari_gpt/checkpoint-3500/ (stored 0%)
  adding: jaydari_gpt/checkpoint-3500/README.md (deflated 65%)
  adding: jaydari_gpt/checkpoint-3500/rng_state.pth (deflated 26%)
  adding: jaydari_gpt/checkpoint-3500/tokenizer.json (deflated 85%)
  adding: jaydari_gpt/checkpoint-3500/tokenizer_config.json (deflated 46%)
  adding: jaydari_gpt/checkpoint-3500/scheduler.pt (deflated 61%)
  adding: jaydari_gpt/checkpoint-3500/optimizer.pt (deflated 22%)
  adding: jaydari_gpt/checkpoint-3500/trainer_state.json (deflated 81%)
  adding: jaydari_gpt/checkpoint-3500/training_args.bin (deflated 53%)
  adding: jaydari_gpt/checkpoint-3500/adapter_model.safetensors (deflated 23%)
  adding: jaydari_gpt/checkpoint-3500/chat_template.jinja (deflated 60%)
  adding: jaydari_gpt/checkpoint-3500/adapter_config.jso

In [ ]:
from google.colab import files
files.download("jaydari_gpt.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
#######################
#   model testing     #
#######################